# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 11.2 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference

In [5]:
TASK_ID = "task286"
SOURCE_JSON_NAME = "task286(1).json"
RULE_DESCRIPTION = "maze connected-component flood fill from colored seed, with two-color checkerboard parity through passable non-wall cells"
CH = 10
H = W = 30
candidate_paths = [
    Path.cwd() / SOURCE_JSON_NAME,
    Path.cwd() / f"{TASK_ID}.json",
    Path("/mnt/data") / SOURCE_JSON_NAME,
    Path("/mnt/data") / f"{TASK_ID}.json",
    Path(COMPETITION) / SOURCE_JSON_NAME,
    Path(COMPETITION) / f"{TASK_ID}.json",
]
TASK_JSON = next((p for p in candidate_paths if p.exists()), None)
assert TASK_JSON is not None, "Could not locate task JSON. Checked: " + ", ".join(str(p) for p in candidate_paths)
OUT_DIR = Path.cwd() / f"{TASK_ID}_static_onnx"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ONNX_PATH = OUT_DIR / f"{TASK_ID}.onnx"
SUBMISSION_PATH = Path.cwd() / "submission.zip"
SUMMARY_PATH = OUT_DIR / f"{TASK_ID}_validation_summary.json"

with TASK_JSON.open("r") as f:
    task = json.load(f)

print("task json:", TASK_JSON)
len(task["train"]), len(task["test"]), len(task.get("arc-gen", []))

task json: /kaggle/input/competitions/neurogolf-2026/task286.json


(2, 1, 262)

In [6]:

def grid_to_tensor_zero_padded(grid, h=H, w=W, ch=CH):
    """Convert ARC grid to [1,10,30,30].
    Inside real grid: one-hot, including background color 0.
    Outside real grid: all-zero across all channels, not color-0 background.
    """
    x = np.zeros((1, ch, h, w), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, v in enumerate(row):
            x[0, int(v), r, c] = 1.0
    return x

def show_grid(g):
    return "\n".join("".join(str(v) if v else "." for v in row) for row in g)


def python_rule(grid):
    arr = np.array(grid, dtype=np.int64)
    h, w = arr.shape
    out = arr.copy()
    markers = (arr != 0) & (arr != 8)
    colors = sorted(set(arr[markers].tolist()))
    if not colors:
        return out.tolist()

    # Learn checkerboard color by coordinate parity from the seed pixels.
    parity_color = {}
    for r, c in zip(*np.where(markers)):
        parity_color[(r + c) & 1] = int(arr[r, c])
    if len(colors) >= 2:
        for p in [0, 1]:
            if p not in parity_color:
                parity_color[p] = [v for v in colors if v not in parity_color.values()][0]
    else:
        parity_color.setdefault(0, colors[0])
        parity_color.setdefault(1, colors[0])

    # Flood-fill the maze component containing the colored seed. Walls are color 8.
    passable = arr != 8
    connected = markers.copy()
    for _ in range(h * w):
        old = connected.copy()
        nbr = connected.copy()
        nbr[1:, :] |= connected[:-1, :]
        nbr[:-1, :] |= connected[1:, :]
        nbr[:, 1:] |= connected[:, :-1]
        nbr[:, :-1] |= connected[:, 1:]
        connected = nbr & passable
        if np.array_equal(old, connected):
            break

    for r, c in zip(*np.where(connected & passable)):
        out[r, c] = parity_color[(r + c) & 1]
    return out.tolist()


for split in ["train", "test", "arc-gen"]:
    ok = sum(python_rule(ex["input"]) == ex["output"] for ex in task.get(split, []))
    print(split, ok, "/", len(task.get(split, [])))


train 2 / 2
test 1 / 1
arc-gen 262 / 262


In [7]:

class Base(nn.Module):
    def __init__(self, h=H, w=W):
        super().__init__()
        rr = torch.arange(h, dtype=torch.float32).view(1, 1, h, 1).expand(1, 1, h, w)
        cc = torch.arange(w, dtype=torch.float32).view(1, 1, 1, w).expand(1, 1, h, w)
        self.register_buffer("R", rr)
        self.register_buffer("C", cc)

    def color_map(self, x):
        return torch.argmax(x, dim=1, keepdim=True).float()

    def active(self, x):
        # Real ARC grid cells have exactly one active channel, including color-0 background.
        # Padding outside the real grid has all-zero channels and is excluded.
        return (x.sum(dim=1, keepdim=True) > 0.5).float()

    def to_onehot(self, colors, active):
        outs = []
        for k in range(10):
            outs.append(((colors - float(k)).abs() < 0.25).float() * active)
        return torch.cat(outs, dim=1)

class Task286Model(Base):
    def __init__(self, steps=256):
        super().__init__()
        self.steps = steps
        parity = torch.remainder(self.R + self.C, 2.0)
        self.register_buffer("even", (parity < 0.5).float())
        self.register_buffer("odd", (parity > 0.5).float())

    def forward(self, x):
        active = self.active(x)
        colors = self.color_map(x)

        wall = (x[:, 8:9, :, :] > 0.5).float() * active
        passable = active * (1.0 - wall)
        markers = passable * (1.0 - (x[:, 0:1, :, :] > 0.5).float())

        cnt_even = (markers * self.even).sum(dim=(2, 3), keepdim=True)
        cnt_odd = (markers * self.odd).sum(dim=(2, 3), keepdim=True)
        even_color = (colors * markers * self.even).sum(dim=(2, 3), keepdim=True) / (cnt_even + (cnt_even < 0.5).float())
        odd_color = (colors * markers * self.odd).sum(dim=(2, 3), keepdim=True) / (cnt_odd + (cnt_odd < 0.5).float())

        # Static unrolled 4-neighbor flood fill. This exports as Slice/Cat/Add/Greater,
        # not ONNX Loop/Scan, and 256 steps comfortably covers the generated 30x30 task family.
        fill = markers
        for _ in range(self.steps):
            zr = torch.zeros_like(fill[:, :, :1, :])
            zc = torch.zeros_like(fill[:, :, :, :1])
            up = torch.cat([zr, fill[:, :, :-1, :]], dim=2)
            down = torch.cat([fill[:, :, 1:, :], zr], dim=2)
            left = torch.cat([zc, fill[:, :, :, :-1]], dim=3)
            right = torch.cat([fill[:, :, :, 1:], zc], dim=3)
            fill = ((fill + up + down + left + right) > 0.5).float() * passable

        parity_color = even_color * self.even + odd_color * self.odd
        out = torch.where(fill > 0.5, parity_color, colors)
        return self.to_onehot(out, active)

model = Task286Model(steps=256).eval()


In [8]:

dummy = torch.from_numpy(grid_to_tensor_zero_padded(task["test"][0]["input"]))

torch.onnx.export(
    model,
    dummy,
    str(ONNX_PATH),
    input_names=["input"],
    output_names=["output"],
    opset_version=17,
    do_constant_folding=True,
    dynamic_axes=None,
    dynamo=False,
)

onnx_model = onnx.load(str(ONNX_PATH))
onnx_model = onnx.shape_inference.infer_shapes(onnx_model)
onnx.save(onnx_model, str(ONNX_PATH))
onnx.checker.check_model(str(ONNX_PATH))

ONNX_PATH, ONNX_PATH.stat().st_size


/tmp/ipykernel_16/2818671719.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


(PosixPath('/kaggle/working/task286_static_onnx/task286.onnx'), 1385229)

In [9]:

def vi_shape(vi):
    dims = []
    for d in vi.type.tensor_type.shape.dim:
        if d.dim_value:
            dims.append(int(d.dim_value))
        elif d.dim_param:
            dims.append(str(d.dim_param))
        else:
            dims.append(None)
    return dims

onnx_model = onnx.load(str(ONNX_PATH))
ops = collections.Counter(node.op_type for node in onnx_model.graph.node)
forbidden = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}
empty_inputs = [
    (node.name, node.op_type, list(node.input))
    for node in onnx_model.graph.node
    if any(inp == "" for inp in node.input)
]
bad_shapes = []
for vi in list(onnx_model.graph.input) + list(onnx_model.graph.value_info) + list(onnx_model.graph.output):
    shp = vi_shape(vi)
    if any(d is None or isinstance(d, str) for d in shp):
        bad_shapes.append((vi.name, shp))

print("input shape:", vi_shape(onnx_model.graph.input[0]))
print("output shape:", vi_shape(onnx_model.graph.output[0]))
print("ONNX size:", ONNX_PATH.stat().st_size)
print("ops:", dict(ops))
print("forbidden ops:", sorted(forbidden & set(ops)))
print("empty optional inputs:", len(empty_inputs))
print("non-static tensor shapes:", len(bad_shapes))

assert vi_shape(onnx_model.graph.input[0]) == [1, 10, 30, 30]
assert vi_shape(onnx_model.graph.output[0]) == [1, 10, 30, 30]
assert not (forbidden & set(ops))
assert not empty_inputs
assert not bad_shapes
assert ONNX_PATH.stat().st_size < 1_400_000


input shape: [1, 10, 30, 30]
output shape: [1, 10, 30, 30]
ONNX size: 1385229
ops: {'Constant': 5414, 'ReduceSum': 5, 'Greater': 260, 'Cast': 272, 'ArgMax': 1, 'Slice': 1026, 'Mul': 276, 'Sub': 12, 'Less': 12, 'Add': 1027, 'Div': 2, 'Concat': 1025, 'Where': 1, 'Abs': 10}
forbidden ops: []
empty optional inputs: 0
non-static tensor shapes: 0


In [10]:

sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])

def validate_examples(examples):
    tensor_ok = 0
    grid_ok = 0
    outside_zero_ok = 0
    bad = []
    for i, ex in enumerate(examples):
        x = grid_to_tensor_zero_padded(ex["input"])
        y = sess.run(None, {"input": x})[0]
        exp = grid_to_tensor_zero_padded(ex["output"])
        pred_bin = (y > 0.5).astype(np.float32)

        if np.array_equal(pred_bin, exp):
            tensor_ok += 1
        else:
            bad.append(i)

        h, w = len(ex["output"]), len(ex["output"][0])
        pred_grid = pred_bin[0, :, :h, :w].argmax(axis=0).astype(np.int64).tolist()
        if pred_grid == ex["output"]:
            grid_ok += 1

        active = x.sum(axis=1, keepdims=True) > 0.5
        if np.all(np.abs(y * (~active)) < 1e-5):
            outside_zero_ok += 1

    return {
        "tensor_exact_zero_padded": [tensor_ok, len(examples)],
        "grid_argmax_inside_canvas": [grid_ok, len(examples)],
        "outside_active_all_channels_zero": [outside_zero_ok, len(examples)],
        "bad_indices": bad[:10],
    }

def deterministic_holdout(examples, fraction=0.60):
    idx = list(range(len(examples)))
    rng = random.Random(20260707)
    rng.shuffle(idx)
    n = int(math.ceil(len(idx) * fraction))
    return [examples[i] for i in idx[:n]], idx[:n]

arc_holdout, arc_holdout_indices = deterministic_holdout(task.get("arc-gen", []), 0.60)
summary = {
    "task_id": TASK_ID,
    "rule": RULE_DESCRIPTION,
    "onnx_path": str(ONNX_PATH),
    "onnx_size_bytes": ONNX_PATH.stat().st_size,
    "input_shape": vi_shape(onnx_model.graph.input[0]),
    "output_shape": vi_shape(onnx_model.graph.output[0]),
    "ops": dict(ops),
    "forbidden_ops": sorted(forbidden & set(ops)),
    "empty_optional_inputs": len(empty_inputs),
    "non_static_tensor_shapes": len(bad_shapes),
    "arc_gen_holdout_fraction": 0.60,
    "arc_gen_holdout_count": len(arc_holdout),
    "arc_gen_holdout_indices_first_20": arc_holdout_indices[:20],
    "validation": {
        "train": validate_examples(task["train"]),
        "test": validate_examples(task["test"]),
        "arc-gen-60pct-holdout": validate_examples(arc_holdout),
        "arc-gen-full": validate_examples(task.get("arc-gen", [])),
    },
}

print(json.dumps(summary, indent=2)[:5000])
with SUMMARY_PATH.open("w") as f:
    json.dump(summary, f, indent=2)


{
  "task_id": "task286",
  "rule": "maze connected-component flood fill from colored seed, with two-color checkerboard parity through passable non-wall cells",
  "onnx_path": "/kaggle/working/task286_static_onnx/task286.onnx",
  "onnx_size_bytes": 1385229,
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "ops": {
    "Constant": 5414,
    "ReduceSum": 5,
    "Greater": 260,
    "Cast": 272,
    "ArgMax": 1,
    "Slice": 1026,
    "Mul": 276,
    "Sub": 12,
    "Less": 12,
    "Add": 1027,
    "Div": 2,
    "Concat": 1025,
    "Where": 1,
    "Abs": 10
  },
  "forbidden_ops": [],
  "empty_optional_inputs": 0,
  "non_static_tensor_shapes": 0,
  "arc_gen_holdout_fraction": 0.6,
  "arc_gen_holdout_count": 158,
  "arc_gen_holdout_indices_first_20": [
    115,
    162,
    20,
    138,
    9,
    13,
    136,
    163,
    248,
    146,
    144,
    55,
    127,
    126,
    28,
    35,
    227,
    216,
    239,
    241
  ],
  "v

In [11]:

with zipfile.ZipFile(SUBMISSION_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(ONNX_PATH, arcname=f"{TASK_ID}.onnx")

print("Wrote:", SUBMISSION_PATH)
print("Zip contents:", zipfile.ZipFile(SUBMISSION_PATH).namelist())
assert zipfile.ZipFile(SUBMISSION_PATH).namelist() == [f"{TASK_ID}.onnx"]


Wrote: /kaggle/working/submission.zip
Zip contents: ['task286.onnx']
